In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import cdist

c:\Users\Hemanth Sai\OneDrive\Desktop\info\queytube\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load cleaned data
df = pd.read_csv("cleaned_transcripts.csv")

# Load query-video mapping
mapping_df = pd.read_csv("query_video_map.csv")

# Prepare queries and ground truth
queries = mapping_df["query"].tolist()
ground_truth = dict(zip(mapping_df["query"], mapping_df["relevant_video_id"]))

# Step 3: Keep title and transcript separate for individual embeddings
# Also create combined text for joint encoding
titles = df["title"].tolist()
transcripts = df["transcript"].tolist()
df["text"] = df["title"] + " " + df["transcript"]
documents = df["text"].tolist()

video_ids = df["video_id"].tolist()

print("Data loaded successfully")
print("Total videos:", len(video_ids))
print("Total queries:", len(queries))

Data loaded successfully
Total videos: 345
Total queries: 77


In [3]:
models = {
    "MiniLM": "all-MiniLM-L6-v2",
    "MPNet": "all-mpnet-base-v2",
    "MultiQA": "multi-qa-MiniLM-L6-cos-v1"
}

In [4]:
results = []

for model_name, model_path in models.items():

    print(f"\nLoading model: {model_name}")
    model = SentenceTransformer(model_path)

    # Step 3: Generate embeddings for titles and transcripts separately
    title_embeddings = model.encode(titles, convert_to_numpy=True)
    transcript_embeddings = model.encode(transcripts, convert_to_numpy=True)
    print(f"  Title embeddings shape: {title_embeddings.shape}")
    print(f"  Transcript embeddings shape: {transcript_embeddings.shape}")

    # Combine title + transcript embeddings (average) for ranking
    doc_embeddings = (title_embeddings + transcript_embeddings) / 2

    # Step 4: Query embeddings
    query_embeddings = model.encode(queries, convert_to_numpy=True)

    # Step 5 & 6: Metrics
    metrics = {
        "cosine": lambda q: cosine_similarity(q, doc_embeddings),
        "dot": lambda q: np.dot(q, doc_embeddings.T),
        "euclidean": lambda q: -cdist(q, doc_embeddings, metric="euclidean")
    }

    for metric_name, metric_func in metrics.items():

        print(f"\nEvaluating {model_name} with {metric_name}")

        scores = metric_func(query_embeddings)

        top1 = 0
        top3 = 0
        top5 = 0
        ranks = []

        detailed_results = []

        # ================================
        # Step 7: Ranking
        # ================================

        for i, query in enumerate(queries):
            score = scores[i]

            ranked_idx = np.argsort(score)[::-1]
            ranked_videos = [video_ids[j] for j in ranked_idx]

            true_video = ground_truth[query]

            # Find rank of correct video
            if true_video in ranked_videos:
                rank = ranked_videos.index(true_video) + 1
            else:
                rank = -1

            # Store Top 5 results
            for r in range(5):
                detailed_results.append({
                    "query": query,
                    "rank": r + 1,
                    "video_id": ranked_videos[r],
                    "score": score[ranked_idx[r]]
                })

            # Print per-query result
            print(f"\nQuery: {query}")
            print(f"Expected Video: {true_video}")
            print(f"Retrieved Rank: {rank}")

            ranks.append(rank if rank != -1 else len(video_ids))

            if rank == 1:
                top1 += 1
            if rank != -1 and rank <= 3:
                top3 += 1
            if rank != -1 and rank <= 5:
                top5 += 1

        # ================================
        # Step 8: Evaluation Metrics
        # ================================

        total = len(queries)

        print("\n===== Evaluation Metrics =====")
        print(f"Model: {model_name}")
        print(f"Metric: {metric_name}")
        print(f"Top-1 Recall: {top1 / total:.3f}")
        print(f"Top-3 Recall: {top3 / total:.3f}")
        print(f"Top-5 Recall: {top5 / total:.3f}")
        print(f"Average Rank: {np.mean(ranks):.2f}")

        # Save summary
        results.append({
            "Model": model_name,
            "Metric": metric_name,
            "Top-1 Recall": top1 / total,
            "Top-3 Recall": top3 / total,
            "Top-5 Recall": top5 / total,
            "Avg Rank": np.mean(ranks)
        })

        # Save detailed results (Step 7 output)
        detailed_df = pd.DataFrame(detailed_results)
        detailed_df.to_csv(f"detailed_results_{model_name}_{metric_name}.csv", index=False)


Loading model: MiniLM


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1584.52it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Title embeddings shape: (345, 384)
  Transcript embeddings shape: (345, 384)

Evaluating MiniLM with cosine

Query: How do you build a video sharing app using Next.js?
Expected Video: IBTx5aGj-6U
Retrieved Rank: 1

Query: What are closures in JavaScript?
Expected Video: U6-RekkuORI
Retrieved Rank: 1

Query: How do regular expressions work in programming?
Expected Video: gmuTjeQUbTM
Retrieved Rank: 64

Query: What is integration testing in software development?
Expected Video: s950xhRvqYQ
Retrieved Rank: 1

Query: How do you contribute to open source projects?
Expected Video: cw02dMpWStI
Retrieved Rank: 3

Query: What is Git and why is it important?
Expected Video: 0WjfKQdfeMU
Retrieved Rank: 279

Query: How do developers collaborate using GitHub?
Expected Video: gmuTjeQUbTM
Retrieved Rank: 266

Query: What is Kubernetes used for?
Expected Video: 0WjfKQdfeMU
Retrieved Rank: 64

Query: How does Kubernetes manage containers?
Expected Video: 0WjfKQdfeMU
Retrieved Rank: 98

Query: What is

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 904.49it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Title embeddings shape: (345, 768)
  Transcript embeddings shape: (345, 768)

Evaluating MPNet with cosine

Query: How do you build a video sharing app using Next.js?
Expected Video: IBTx5aGj-6U
Retrieved Rank: 1

Query: What are closures in JavaScript?
Expected Video: U6-RekkuORI
Retrieved Rank: 1

Query: How do regular expressions work in programming?
Expected Video: gmuTjeQUbTM
Retrieved Rank: 13

Query: What is integration testing in software development?
Expected Video: s950xhRvqYQ
Retrieved Rank: 1

Query: How do you contribute to open source projects?
Expected Video: cw02dMpWStI
Retrieved Rank: 4

Query: What is Git and why is it important?
Expected Video: 0WjfKQdfeMU
Retrieved Rank: 200

Query: How do developers collaborate using GitHub?
Expected Video: gmuTjeQUbTM
Retrieved Rank: 153

Query: What is Kubernetes used for?
Expected Video: 0WjfKQdfeMU
Retrieved Rank: 15

Query: How does Kubernetes manage containers?
Expected Video: 0WjfKQdfeMU
Retrieved Rank: 24

Query: What is 

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 859.39it/s]
BertModel LOAD REPORT from: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Title embeddings shape: (345, 384)
  Transcript embeddings shape: (345, 384)

Evaluating MultiQA with cosine

Query: How do you build a video sharing app using Next.js?
Expected Video: IBTx5aGj-6U
Retrieved Rank: 1

Query: What are closures in JavaScript?
Expected Video: U6-RekkuORI
Retrieved Rank: 1

Query: How do regular expressions work in programming?
Expected Video: gmuTjeQUbTM
Retrieved Rank: 203

Query: What is integration testing in software development?
Expected Video: s950xhRvqYQ
Retrieved Rank: 1

Query: How do you contribute to open source projects?
Expected Video: cw02dMpWStI
Retrieved Rank: 5

Query: What is Git and why is it important?
Expected Video: 0WjfKQdfeMU
Retrieved Rank: 302

Query: How do developers collaborate using GitHub?
Expected Video: gmuTjeQUbTM
Retrieved Rank: 276

Query: What is Kubernetes used for?
Expected Video: 0WjfKQdfeMU
Retrieved Rank: 66

Query: How does Kubernetes manage containers?
Expected Video: 0WjfKQdfeMU
Retrieved Rank: 70

Query: What 

In [5]:
results_df = pd.DataFrame(results)

print("\n===== FINAL COMPARISON =====\n")
print(results_df)

# Save results
results_df.to_csv("embedding_evaluation_results.csv", index=False)


===== FINAL COMPARISON =====

     Model     Metric  Top-1 Recall  Top-3 Recall  Top-5 Recall    Avg Rank
0   MiniLM     cosine      0.051948      0.077922      0.077922  124.818182
1   MiniLM        dot      0.051948      0.064935      0.077922  124.844156
2   MiniLM  euclidean      0.051948      0.077922      0.103896  135.012987
3    MPNet     cosine      0.051948      0.064935      0.116883   96.246753
4    MPNet        dot      0.051948      0.064935      0.116883   94.779221
5    MPNet  euclidean      0.051948      0.077922      0.116883  111.532468
6  MultiQA     cosine      0.051948      0.077922      0.090909  115.285714
7  MultiQA        dot      0.051948      0.077922      0.103896  112.766234
8  MultiQA  euclidean      0.064935      0.077922      0.090909  133.649351


In [6]:
# ================================
# Step 10: Identify Best Model and Ranking Method
# ================================

results_df = pd.DataFrame(results)

# Find the row with highest Top-3 Recall (primary criterion)
# Break ties by lowest Avg Rank (secondary criterion)
best_idx = results_df.sort_values(
    by=["Top-3 Recall", "Avg Rank"],
    ascending=[False, True]
).index[0]

best_row = results_df.loc[best_idx]

best_model  = best_row["Model"]
best_metric = best_row["Metric"]
best_top1   = best_row["Top-1 Recall"]
best_top3   = best_row["Top-3 Recall"]
best_top5   = best_row["Top-5 Recall"]
best_rank   = best_row["Avg Rank"]

print("===== STEP 10: BEST MODEL & RANKING METHOD =====")
print(f"Best Model  : {best_model}")
print(f"Best Metric : {best_metric}")
print()
print(f"Top-1 Recall : {best_top1:.3f}")
print(f"Top-3 Recall : {best_top3:.3f}")
print(f"Top-5 Recall : {best_top5:.3f}")
print(f"Average Rank : {best_rank:.2f}")
print()
print("Reason: Selected based on highest Top-3 Recall.")
print(f"        {best_model} with {best_metric} achieves the best balance of")
print("        recall at rank 3 and lowest average rank among all combinations.")

===== STEP 10: BEST MODEL & RANKING METHOD =====
Best Model  : MPNet
Best Metric : euclidean

Top-1 Recall : 0.052
Top-3 Recall : 0.078
Top-5 Recall : 0.117
Average Rank : 111.53

Reason: Selected based on highest Top-3 Recall.
        MPNet with euclidean achieves the best balance of
        recall at rank 3 and lowest average rank among all combinations.
